# 🏥 HealthBridge Medical Center
## Data Cleaning Script — UCI Diabetes 130-US Hospitals Dataset

**Purpose:** Clean the raw UCI dataset so it is ready for the preprocessing notebook.

**Run this notebook FIRST, before the preprocessing notebook.**

**Output:** `diabetic_clean.csv` — a fully cleaned file ready for preprocessing.

---
### What this notebook fixes:
| Issue | Fix |
|---|---|
| `?` used as missing value placeholder | Replace with `NaN` |
| `weight` column — 97% missing | Drop the column |
| `payer_code` column — 40% missing | Drop the column |
| `race`, `medical_specialty`, `diag_1/2/3` — partial missing | Fill with `'Unknown'` |
| Death / hospice discharge records | Remove rows |
| Multiple encounters per patient | Keep first encounter only |
| `Unknown/Invalid` gender records | Remove rows |
| ID columns stored as integers | Convert to `category` dtype |

---

## CELL 1 — Upload your dataset

**What:** Load the raw CSV file into this Colab session.

**Why:** Before we can clean anything, Python needs to read the file.

**Choose ONE of the two options below** depending on how you stored your file.

> ✅ **Option A** — Upload from your computer (easiest first time)  
> ✅ **Option B** — Load from Google Drive (best for ongoing work)

In [ ]:
# ============================================================
# OPTION A — Upload directly from your computer
# Run this cell → a 'Choose Files' button will appear
# Click it and select diabetic_data.csv
# ============================================================

from google.colab import files
uploaded = files.upload()

# After uploading, the file is available as 'diabetic_data.csv'
FILE_PATH = 'diabetic_data.csv'

print('✅ File uploaded successfully!')
print(f'   File name: {list(uploaded.keys())[0]}')

In [ ]:
# ============================================================
# OPTION B — Load from Google Drive (recommended)
# Only run this cell if you chose Option B — skip if you used Option A
# 
# Before running: upload diabetic_data.csv to your Google Drive
# Update the FILE_PATH below to match where you saved it
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to match your Google Drive folder location
FILE_PATH = '/content/drive/MyDrive/diabetic_data.csv'

# Example if you put it in a folder called 'HealthBridge':
# FILE_PATH = '/content/drive/MyDrive/HealthBridge/diabetic_data.csv'

print(f'✅ Drive mounted. Will load from: {FILE_PATH}')

---
## CELL 2 — Import libraries and load the data

**What:** Load Python tools and read the CSV file into a DataFrame.

**Why:** `pandas` is how Python works with tables. A DataFrame is like a spreadsheet inside Python — rows are patient encounters, columns are variables.

**What to look for:** The shape should say `(101766, 50)` — 101,766 rows and 50 columns.

In [ ]:
import pandas as pd
import numpy as np

# Display settings — show more columns and cleaner numbers
pd.set_option('display.max_columns', 55)
pd.set_option('display.float_format', '{:.1f}'.format)

# ── Load the raw data ─────────────────────────────────────────────────────────
df = pd.read_csv(FILE_PATH)

print('=' * 55)
print('  RAW DATASET LOADED')
print('=' * 55)
print(f'  Rows:    {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')
print()
print('Column names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

In [ ]:
# ── Quick look at the first 5 rows ────────────────────────────────────────────
# This gives you a feel for what the raw data looks like
# Notice the '?' values scattered throughout — we fix those in the next step
print('First 5 rows of raw data:')
df.head()

---
## CELL 3 — Fix #1: Replace `?` with proper missing values

**What:** Find every `?` in the entire dataset and replace it with `NaN`.

**Why:** This dataset was built in the late 1990s/early 2000s when `?` was a common way to mark missing data. Python doesn't know `?` means 'missing' — it treats it as regular text. If we leave them in, every downstream calculation will be silently wrong.

`NaN` (Not a Number) is Python's official way of saying 'this value is missing'.

**What to look for:** The count after replacement should be 0.

In [ ]:
# Count how many ? exist before replacing
q_before = (df == '?').sum().sum()
print(f'Question marks BEFORE replacement: {q_before:,}')

# Show WHICH columns have ? values
print('\nColumns with ? values:')
q_by_col = (df == '?').sum()
q_by_col = q_by_col[q_by_col > 0].sort_values(ascending=False)
for col, count in q_by_col.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {col:<30} {count:>7,}  ({pct:5.1f}%)  {bar}')

# ── Replace ALL ? with NaN ────────────────────────────────────────────────────
df.replace('?', np.nan, inplace=True)

# Confirm replacement worked
q_after = (df == '?').sum().sum()
print(f'\nQuestion marks AFTER replacement: {q_after}')

if q_after == 0:
    print('✅ All ? values replaced with NaN')
else:
    print('⚠️  Some ? values remain — check manually')

---
## CELL 4 — Fix #2: Handle missing values column by column

**What:** Decide what to do with each column that has missing values — either drop the whole column or fill the gaps.

**Why:** There are two strategies:
- **Drop** a column when it's so incomplete that keeping it would mislead the model
- **Fill** a column with `'Unknown'` when the column is still useful — this preserves the row while being honest that the value was missing

**Decision rule used here:**
- `weight` (97% missing) → **Drop** — almost no data exists
- `payer_code` (40% missing) → **Drop** — too many gaps, low predictive value
- `medical_specialty`, `diag_1/2/3`, `race` → **Fill** with `'Unknown'` — still useful columns

In [ ]:
# ── Show all missing value counts before any treatment ────────────────────────
print('Missing values by column (before treatment):')
print('-' * 50)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Only show columns that actually have missing values
missing_df = missing_df[missing_df['Missing Count'] > 0]

for col, row in missing_df.iterrows():
    action = '→ DROP' if row['Missing %'] > 30 else '→ FILL with Unknown'
    print(f'  {col:<30} {row["Missing Count"]:>7,}  ({row["Missing %"]:5.1f}%)  {action}')

print(f'\nTotal missing cells: {missing_df["Missing Count"].sum():,}')

In [ ]:
# ── DROP columns with too much missing data ───────────────────────────────────

cols_to_drop = ['weight', 'payer_code']

# Only drop if they exist (in case you already ran this cell once)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df.drop(columns=cols_to_drop, inplace=True)

print('Dropped columns:')
for col in cols_to_drop:
    print(f'  🗑️  {col}')

print(f'\nDataset shape after dropping: {df.shape}')

In [ ]:
# ── FILL remaining missing columns with 'Unknown' ────────────────────────────
# This keeps the row (patient encounter) while honestly marking the gap

cols_to_fill = ['race', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']

for col in cols_to_fill:
    if col in df.columns:
        before = df[col].isnull().sum()
        df[col] = df[col].fillna('Unknown')
        after = df[col].isnull().sum()
        print(f'  ✏️  {col:<25} filled {before:,} gaps  (nulls remaining: {after})')

# ── Verify: check remaining missing values ────────────────────────────────────
remaining = df.isnull().sum().sum()
print(f'\nTotal missing values remaining: {remaining}')

if remaining == 0:
    print('✅ No missing values left!')
else:
    print('⚠️  Remaining missing values:')
    print(df.isnull().sum()[df.isnull().sum() > 0])

---
## CELL 5 — Fix #3: Remove records that cannot be readmitted

**What:** Remove patient records where the patient died during their stay or was sent to hospice.

**Why:** Our goal is to predict 30-day readmission. If a patient died in the hospital or was discharged to hospice, they cannot physically be readmitted 30 days later. Including these records would add noise that corrupts our model — the target variable `readmitted` would be meaningless for these rows.

These are flagged by specific values in `discharge_disposition_id`:
- `11` = Expired (died in hospital)
- `13` = Hospice — medical facility
- `14` = Hospice — home
- `19` = Expired in a medical facility
- `20` = Expired at home
- `21` = Expired — destination unknown

In [ ]:
rows_before = len(df)

# IDs that mean the patient cannot be readmitted
cant_be_readmitted = [11, 13, 14, 19, 20, 21]

# Count how many will be removed
to_remove = df['discharge_disposition_id'].isin(cant_be_readmitted).sum()
print(f'Records to remove (death/hospice): {to_remove:,} ({to_remove/len(df)*100:.1f}%)')

# Remove them
# The ~ means 'NOT in this list' — keep everything that is NOT in the remove list
df = df[~df['discharge_disposition_id'].isin(cant_be_readmitted)]

rows_after = len(df)

print(f'\nRows before: {rows_before:,}')
print(f'Rows after:  {rows_after:,}')
print(f'Removed:     {rows_before - rows_after:,} records')
print('✅ Death and hospice records removed')

---
## CELL 6 — Fix #4: Keep only one encounter per patient

**What:** The dataset contains multiple rows for the same patient (multiple hospital visits). We keep only the **first** visit per patient.

**Why — Data Leakage:** This is one of the most important concepts in ML. If the same patient appears in both our training set and test set, the model can "cheat" by learning patterns from the same patient twice. That would make the model look better in testing than it actually is in real life.

By keeping only the first encounter per patient, we ensure each patient appears exactly once, making the results honest and realistic.

In [ ]:
rows_before = len(df)
unique_patients_before = df['patient_nbr'].nunique()

# How many patients have multiple encounters?
encounter_counts = df['patient_nbr'].value_counts()
multi_encounter = (encounter_counts > 1).sum()
print(f'Patients with multiple encounters: {multi_encounter:,}')
print(f'Max encounters for one patient:   {encounter_counts.max()}')
print()

# Sort by encounter_id (ascending) so the earliest encounter comes first
# Then drop_duplicates keeps only the first row for each patient_nbr
df = df.sort_values('encounter_id', ascending=True)
df = df.drop_duplicates(subset='patient_nbr', keep='first')

rows_after = len(df)
unique_patients_after = df['patient_nbr'].nunique()

print(f'Rows before deduplication: {rows_before:,}')
print(f'Rows after deduplication:  {rows_after:,}')
print(f'Rows removed:              {rows_before - rows_after:,}')
print(f'Unique patients now:       {unique_patients_after:,}')
print(f'Duplicate check:           {df["patient_nbr"].duplicated().sum()} (should be 0)')
print('✅ One encounter per patient — no data leakage')

---
## CELL 7 — Fix #5: Remove invalid gender records

**What:** Remove the tiny group of records with `gender = 'Unknown/Invalid'`.

**Why:** There are fewer than 100 of these rows (less than 0.1% of the dataset). We cannot encode them meaningfully as Male or Female, and the sample is too small to treat as its own category. Dropping them has essentially zero impact on the model but avoids encoding confusion.

In [ ]:
print('Gender distribution before:')
print(df['gender'].value_counts())

rows_before = len(df)

# Keep only rows where gender is Male or Female
df = df[df['gender'].isin(['Male', 'Female'])]

print(f'\nRemoved {rows_before - len(df)} Unknown/Invalid gender records')
print('\nGender distribution after:')
print(df['gender'].value_counts())
print('✅ Gender column cleaned')

---
## CELL 8 — Fix #6: Correct data types for ID columns

**What:** Convert three columns from `int` (integer) to `category` data type.

**Why:** `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` look like numbers — they contain values like 1, 2, 3, 7. But they are actually **category codes**, not real numbers.

If we leave them as integers, the model might wrongly assume that `7` is "greater than" or "more than" `3` — which has no real-world meaning. Converting to `category` tells pandas (and later, our ML model) to treat them as labels, not quantities.

Think of it like a zip code — 75001 is not "larger" than 10001, it's just a different code.

In [ ]:
id_columns = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']

print('Data types BEFORE fix:')
for col in id_columns:
    if col in df.columns:
        print(f'  {col:<35} dtype: {df[col].dtype}  unique values: {df[col].nunique()}')

# Convert to category
for col in id_columns:
    if col in df.columns:
        df[col] = df[col].astype('category')

print('\nData types AFTER fix:')
for col in id_columns:
    if col in df.columns:
        print(f'  {col:<35} dtype: {df[col].dtype}  unique values: {df[col].nunique()}')

print('✅ ID columns converted to category dtype')

---
## CELL 9 — Fix #7: Strip whitespace from text columns

**What:** Remove any leading or trailing spaces from all text columns.

**Why:** A value like `' Male'` (space before Male) and `'Male'` look the same to a human but are treated as different values by Python. This can cause silent errors in grouping and encoding. One quick `.str.strip()` call cleans all of them at once.

In [ ]:
# Get all text (object dtype) columns
text_columns = df.select_dtypes(include='object').columns

print(f'Stripping whitespace from {len(text_columns)} text columns...')

for col in text_columns:
    df[col] = df[col].str.strip()

print(f'✅ Whitespace stripped from: {list(text_columns)}')

---
## CELL 10 — Spot-check numeric columns for impossible values

**What:** Check the min and max of every numeric column to flag anything that looks wrong.

**Why:** Data entry errors can create impossible values — a negative `time_in_hospital`, a `num_medications` of 500, etc. These would skew the model without triggering any Python error. A quick range check catches them before they cause problems.

**What to look for:** Any min/max that seems medically impossible based on what the column represents.

In [ ]:
# Known expected ranges for this dataset (from the UCI documentation)
expected_ranges = {
    'time_in_hospital':    (1, 14),     # Dataset constraint: 1-14 days
    'num_lab_procedures':  (1, 132),    # Documented max
    'num_procedures':      (0, 6),      # Documented max
    'num_medications':     (1, 81),     # Documented max
    'number_diagnoses':    (1, 16),     # Documented max
    'number_outpatient':   (0, 42),     # Historical max observed
    'number_emergency':    (0, 76),     # Historical max observed
    'number_inpatient':    (0, 21),     # Historical max observed
}

print(f'{"Column":<25} {"Min":>6} {"Max":>6} {"Mean":>8} {"Expected Range":<20} {"Status"}')
print('-' * 80)

issues_found = 0
for col, (exp_min, exp_max) in expected_ranges.items():
    if col in df.columns:
        actual_min = df[col].min()
        actual_max = df[col].max()
        actual_mean = df[col].mean()

        # Flag if outside expected range
        if actual_min < exp_min or actual_max > exp_max:
            status = '⚠️  CHECK'
            issues_found += 1
        else:
            status = '✅ OK'

        print(f'{col:<25} {actual_min:>6} {actual_max:>6} {actual_mean:>8.1f}'
              f'  ({exp_min}–{exp_max})          {status}')

print()
if issues_found == 0:
    print('✅ All numeric columns are within expected ranges — no impossible values found')
else:
    print(f'⚠️  {issues_found} column(s) outside expected range — review manually')
    print('   Note: Slightly exceeding documented max is usually OK — the documentation')
    print('   may not cover every edge case. Only worry about extreme outliers.')

---
## CELL 11 — Final validation: confirm everything is clean

**What:** Run a complete quality checklist across the cleaned dataset.

**Why:** Before saving, we verify every fix was applied correctly. If anything fails here, go back to the relevant cell and re-run it.

**Every check should show ✅ before you proceed.**

In [ ]:
print('=' * 60)
print('  FINAL CLEANING VALIDATION CHECKLIST')
print('=' * 60)

checks_passed = 0
checks_total  = 0

def check(label, condition, detail=''):
    global checks_passed, checks_total
    checks_total += 1
    status = '✅' if condition else '❌'
    if condition:
        checks_passed += 1
    extra = f'  → {detail}' if detail else ''
    print(f'  {status}  {label}{extra}')

# 1. No ? remaining
q_remaining = (df == '?').sum().sum()
check('No ? values remaining',
      q_remaining == 0,
      f'{q_remaining} found' if q_remaining > 0 else 'all replaced with NaN')

# 2. No NaN remaining
nan_remaining = df.isnull().sum().sum()
check('No NaN values remaining',
      nan_remaining == 0,
      f'{nan_remaining:,} remaining' if nan_remaining > 0 else 'all filled or dropped')

# 3. weight column dropped
check('weight column dropped',
      'weight' not in df.columns,
      'still present — re-run Cell 4' if 'weight' in df.columns else 'confirmed removed')

# 4. payer_code column dropped
check('payer_code column dropped',
      'payer_code' not in df.columns,
      'still present — re-run Cell 4' if 'payer_code' in df.columns else 'confirmed removed')

# 5. No death/hospice records
cant_readmit = [11,13,14,19,20,21]
bad_records = df['discharge_disposition_id'].isin(cant_readmit).sum()
check('No death/hospice discharge records',
      bad_records == 0,
      f'{bad_records:,} still present — re-run Cell 5' if bad_records > 0 else 'confirmed removed')

# 6. No duplicate patients
dup_patients = df['patient_nbr'].duplicated().sum()
check('No duplicate patient encounters',
      dup_patients == 0,
      f'{dup_patients:,} duplicates remain — re-run Cell 6' if dup_patients > 0 else 'one encounter per patient')

# 7. No Unknown/Invalid gender
invalid_gender = (df['gender'] == 'Unknown/Invalid').sum()
check('No Unknown/Invalid gender records',
      invalid_gender == 0,
      f'{invalid_gender} remain — re-run Cell 7' if invalid_gender > 0 else 'confirmed removed')

# 8. ID columns are category dtype
id_cols_ok = all(str(df[c].dtype) == 'category'
                 for c in ['admission_type_id','discharge_disposition_id','admission_source_id']
                 if c in df.columns)
check('ID columns converted to category dtype',
      id_cols_ok,
      'correct dtype' if id_cols_ok else 're-run Cell 8')

# 9. Target variable present
check('Target variable (readmitted) present',
      'readmitted' in df.columns,
      f'Values: {df["readmitted"].value_counts().to_dict()}' if 'readmitted' in df.columns else 'MISSING!')

# 10. Reasonable final row count
reasonable_size = 50000 <= len(df) <= 90000
check('Final row count is reasonable (50k–90k)',
      reasonable_size,
      f'{len(df):,} rows')

print()
print(f'  Result: {checks_passed}/{checks_total} checks passed')
print('=' * 60)

if checks_passed == checks_total:
    print('  🎉 ALL CHECKS PASSED — data is clean and ready to save!')
else:
    failed = checks_total - checks_passed
    print(f'  ⚠️  {failed} check(s) failed — see notes above and re-run those cells')

---
## CELL 12 — Cleaning summary report

**What:** Print a before-and-after summary of every change made.

**Why:** Good data science practice — document what you did to the data. This summary should go into your project report as the 'Data Cleaning' section.

In [ ]:
print('=' * 65)
print('  DATA CLEANING SUMMARY REPORT')
print('  HealthBridge Medical Center — Capstone Project')
print('=' * 65)
print()
print(f'  Original dataset:    101,766 rows × 50 columns')
print(f'  Cleaned dataset:     {len(df):,} rows × {df.shape[1]} columns')
print(f'  Rows removed:        {101766 - len(df):,}')
print(f'  Columns removed:     2 (weight, payer_code)')
print()
print('  Changes made:')
print('  ─────────────────────────────────────────────────────')
print('  1. Replaced all ? placeholders with NaN')
print('     → Enables proper missing value detection')
print()
print('  2. Dropped weight column (96.9% missing)')
print('     → Insufficient data for meaningful imputation')
print()
print('  3. Dropped payer_code column (36.3% missing)')
print('     → Low predictive value, high missing rate')
print()
print('  4. Filled race with Unknown (1.7% missing)')
print('     medical_specialty with Unknown (~49% missing)')
print('     diag_1 / diag_2 / diag_3 with Unknown (partial)')
print('     → Preserves rows, flags gaps honestly')
print()
print('  5. Removed death/hospice discharge records')
print('     → discharge_disposition_id in [11,13,14,19,20,21]')
print('     → Patients cannot be readmitted — invalid target')
print()
print('  6. Deduplicated to first encounter per patient')
print('     → Prevents data leakage between train/test sets')
print()
print('  7. Removed Unknown/Invalid gender records (<100 rows)')
print('     → Too small to encode as separate category')
print()
print('  8. Converted ID columns to category dtype')
print('     → admission_type_id, discharge_disposition_id,')
print('        admission_source_id')
print('     → Prevents model treating codes as quantities')
print()
print('  9. Stripped whitespace from all text columns')
print('     → Prevents silent encoding mismatches')
print()
print('  ─────────────────────────────────────────────────────')
print(f'  Target variable distribution:')
vc = df['readmitted'].value_counts()
for val, count in vc.items():
    pct = count / len(df) * 100
    print(f'    {val:<5} → {count:>6,}  ({pct:.1f}%)')
print()
print('  NEXT STEP: Load diabetic_clean.csv into the')
print('  preprocessing notebook to encode features')
print('  and prepare data for machine learning.')
print('=' * 65)

---
## CELL 13 — Save the clean file

**What:** Save the cleaned DataFrame to a CSV file and download it.

**Why:** We want to save the cleaned version so you never have to repeat these cleaning steps again. The preprocessing notebook will load `diabetic_clean.csv` directly.

**Choose the save option that matches how you loaded the file** (Option A or B from Cell 1).

In [ ]:
# ── Save the cleaned CSV ──────────────────────────────────────────────────────
# index=False means don't write the row numbers as a column
df.to_csv('diabetic_clean.csv', index=False)
print(f'✅ Saved diabetic_clean.csv  ({len(df):,} rows × {df.shape[1]} columns)')

In [ ]:
# ============================================================
# OPTION A — Download to your computer
# Run this if you uploaded the file directly (Cell 1 Option A)
# ============================================================

from google.colab import files
files.download('diabetic_clean.csv')
print('📥 Download started — check your Downloads folder')

In [ ]:
# ============================================================
# OPTION B — Save directly to Google Drive
# Run this if you loaded from Google Drive (Cell 1 Option B)
# UPDATE the path below to match your Drive folder
# ============================================================

SAVE_PATH = '/content/drive/MyDrive/diabetic_clean.csv'
# Example with subfolder:
# SAVE_PATH = '/content/drive/MyDrive/HealthBridge/diabetic_clean.csv'

df.to_csv(SAVE_PATH, index=False)
print(f'✅ Saved to Google Drive: {SAVE_PATH}')

In [ ]:
# ============================================================
# BONUS — Also save as Excel (.xlsx)
# Useful for reviewing the data in Excel or sharing with
# non-technical stakeholders
# ============================================================

# Install openpyxl if not already available
try:
    import openpyxl
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'openpyxl', '-q'])

df.to_excel('diabetic_clean.xlsx', index=False, engine='openpyxl')
print('✅ Saved diabetic_clean.xlsx')

# Download the Excel file
from google.colab import files
files.download('diabetic_clean.xlsx')
print('📥 Excel download started')

---
## ✅ You're done with data cleaning!

### What you have now:
- `diabetic_clean.csv` — cleaned dataset ready for preprocessing
- `diabetic_clean.xlsx` — same data in Excel format for review

### What was fixed:
| Fix | Impact |
|---|---|
| Replaced all `?` with `NaN` | Enables proper missing value handling |
| Dropped `weight`, `payer_code` | Removed unusable columns |
| Filled `race`, `medical_specialty`, `diag_1/2/3` | Preserved rows, flagged gaps |
| Removed death/hospice records | Target variable is now valid for all rows |
| Deduplicated patients | No data leakage between train/test |
| Removed invalid gender rows | Cleaner encoding |
| Fixed ID column dtypes | Prevents numeric misinterpretation |

### Next step:
Open the **Preprocessing Notebook** (`Hospital_Readmission_Preprocessing.ipynb`) and load your clean file with:

```python
df = pd.read_csv('diabetic_clean.csv')
```

---
*HealthBridge Medical Center | Advanced Data Analytics Capstone*